# تدريب نموذج كود خاص بيك — Qwen2.5-Coder-1.5B + LoRA
# Fine-tune your own code model (Path B)

النوتبوك دي بتعمل **Fine-tune** (تعليم إضافي) لنموذج مفتوح جاهز هو
`Qwen/Qwen2.5-Coder-1.5B-Instruct` — النموذج ده أصلاً بيفهم لغات برمجة كتير
لأنه اتدرب على كود بكميات ضخمة. إحنا **مش** بندرّب من الصفر؛ إحنا بنعلّمه
أسلوب ونوع مهام معين باستخدام تقنية **LoRA / QLoRA** على داتاست تعليمات كود
حقيقية (`ise-uiuc/Magicoder-OSS-Instruct-75K`).

This notebook fine-tunes the open `Qwen/Qwen2.5-Coder-1.5B-Instruct` model
with LoRA/QLoRA on the Magicoder OSS-Instruct code-instruction dataset, then
pushes the resulting adapter to **your** Hugging Face account.

**Requirements / المتطلبات:**
- Runtime must be a **free GPU (T4)**: `Runtime → Change runtime type → T4 GPU`
- A Hugging Face **Write** token saved in Colab Secrets as `HF_TOKEN`
  (see the login cell below — the token is never written in this notebook)
- Cost: **0** — everything runs on free tiers (Colab free + HF free)

> ⚠️ قبل ما تشغّل: اسم المستخدم مثبّت بالفعل على `darklink56` في خلية الإعدادات (Config) — راجعه لو حابب تغيّر حساب
> مستخدم حسابك على Hugging Face.

In [ ]:
# 1) Check the GPU — لازم يطلع لك كارت T4 (أو غيره من كروت Colab المجانية)
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU found! Go to Runtime → Change runtime type → T4 GPU, then run again.")

In [ ]:
# 2) Install dependencies (versions chosen to work together on Colab)
# transformers + trl (SFTTrainer) + peft (LoRA) + accelerate + bitsandbytes (4-bit) + datasets
!pip install -q "transformers>=4.46,<5" "trl>=0.12,<1" "peft>=0.13,<1" "accelerate>=1.0,<2" "bitsandbytes>=0.44" "datasets>=3.0,<4"

import transformers, trl, peft, datasets
print("transformers", transformers.__version__, "| trl", trl.__version__, "| peft", peft.__version__, "| datasets", datasets.__version__)

In [ ]:
# 3) Log in to Hugging Face — التوكن بييجي من أسرار Colab، مش مكتوب هنا أبداً
# Colab: 🔑 (Secrets) on the left → add HF_TOKEN = your Write token from
# https://huggingface.co/settings/tokens  (and enable "Notebook access")
import os

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    from huggingface_hub import login
    login(token=hf_token)
    print("✅ Logged in via Colab Secrets (HF_TOKEN)")
else:
    # Fallback: interactive login (paste the token in the prompt that appears)
    from huggingface_hub import login
    login()
    print("✅ Logged in interactively")

In [ ]:
# اسم المستخدم مثبّت على darklink56 — مفيش تعديل مطلوب هنا
# 4) Config — عدّل هنا بس / Edit only this cell
BASE_MODEL   = "Qwen/Qwen2.5-Coder-1.5B-Instruct"   # النموذج الأساسي (Apache-2.0)
HF_USERNAME  = "darklink56"                    # ⬅️ اسم مستخدمك على Hugging Face
OUTPUT_REPO  = f"{HF_USERNAME}/qwen-coder-1.5b-lora" # الأدابتور هيترفع هنا

DATASET_NAME = "ise-uiuc/Magicoder-OSS-Instruct-75K" # أعمدة: problem / solution
SAMPLE_SIZE  = 3000        # أول تشغيلة: 3000 مثال بس عشان تخلص في جلسة مجانية
NUM_EPOCHS   = 1
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH = 2       # مناسب لذاكرة T4 (16GB) مع تحميل 4-bit
GRAD_ACCUM = 8             # 2 × 8 = effective batch 16
MAX_SEQ_LEN = 1024

print("Base model :", BASE_MODEL)
print("Output repo:", OUTPUT_REPO)
print("Dataset    :", DATASET_NAME, "| samples:", SAMPLE_SIZE)

In [ ]:
# 5) Load the dataset and format it with the model's chat template
# الداتاست أعمدتها problem (المطلوب) و solution (الحل) — بنحوّل كل صف لمحادثة:
# user = problem  →  assistant = solution
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

raw = load_dataset(DATASET_NAME, split=f"train[:{SAMPLE_SIZE}]")
print(raw)

def to_text(example):
    messages = [
        {"role": "user", "content": example["problem"]},
        {"role": "assistant", "content": example["solution"]},
    ]
    # apply_chat_template بيطلع النص بنفس صيغة Qwen (ChatML) + توكن النهاية
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_ds = raw.map(to_text, remove_columns=raw.column_names)
print(train_ds[0]["text"][:600])

## 6) Training — التدريب

- تحميل النموذج بـ **4-bit (QLoRA)** عشان يدخل في ذاكرة T4.
- **LoRA**: r=16, alpha=32, على كل طبقات الـ projections
  (q/k/v/o + gate/up/down) — بنعلّم أدابتور صغير بس، مش النموذج كله.
- أول تشغيلة بـ 3000 مثال و epoch واحدة: توقّع وقت في حدود جلسة Colab
  مجانية واحدة تقريباً (بيختلف حسب طول الأمثلة وزحمة Colab).

In [ ]:
# 6) Train with TRL SFTTrainer + LoRA (QLoRA 4-bit)
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,  # مهم: من غيرها الأدابتور بيتحمل bf16 و GradScaler بتاع fp16 بيقع على T4
    device_map="auto",
)
model.config.use_cache = False  # لازم مع gradient checkpointing

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

sft_config = SFTConfig(
    output_dir="qwen-coder-1.5b-lora-out",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="no",            # هنرفع الأدابتور في الآخر بس
    fp16=True,                     # T4 مفيهاش bf16
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    max_length=MAX_SEQ_LEN,  # TRL الجديدة: max_length بدل max_seq_length
    # dataset_text_field اتشال: الافتراضي في TRL هو "text" أصلاً
    packing=False,
    report_to=[],
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    peft_config=lora_config,
    processing_class=tokenizer,
)

trainer.train()
print("✅ Training finished")

In [ ]:
# 7) Quick test — جرّب النموذج بعد التدريب على 3 لغات مختلفة
import torch

test_prompts = [
    "Write a Python function that returns the nth Fibonacci number using memoization.",
    "Write a JavaScript function that debounces another function by 300ms.",
    "Write a SQL query that returns the top 3 customers by total order amount from tables customers and orders.",
]

model.config.use_cache = True
model.eval()

for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=220, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print("=" * 80)
    print("PROMPT:", prompt)
    print("-" * 80)
    print(answer)

In [ ]:
# 8) Push the adapter + tokenizer to YOUR Hugging Face account
# النتيجة أدابتور LoRA صغير (مش النموذج كله) — بيتحمّل فوق النموذج الأساسي وقت التشغيل
trainer.push_to_hub(OUTPUT_REPO)
tokenizer.push_to_hub(OUTPUT_REPO)
print(f"✅ Adapter pushed to: https://huggingface.co/{OUTPUT_REPO}")

In [ ]:
# 9) (Optional) Save a backup zip to Google Drive — اختياري
# شيل علامات التعليق (#) من السطور الجاية لو عايز نسخة احتياطية على Drive:
#
# from google.colab import drive
# drive.mount("/content/drive")
# !cd qwen-coder-1.5b-lora-out && zip -r /content/drive/MyDrive/qwen-coder-1.5b-lora.zip checkpoint-* adapter_model.safetensors adapter_config.json 2>/dev/null || zip -r /content/drive/MyDrive/qwen-coder-1.5b-lora.zip .
# print("Saved to Drive")
print("Skipped — optional cell. The adapter is already on Hugging Face after cell 8.")

## Troubleshooting — لو حاجة وقفت

| المشكلة | الحل |
|---|---|
| **CUDA out of memory (OOM)** | قلّل `PER_DEVICE_BATCH` لـ 1 وزوّد `GRAD_ACCUM` لـ 16، أو قلّل `MAX_SEQ_LEN` لـ 768، أو قلّل `SAMPLE_SIZE`. |
| **Runtime disconnected / session ended** | Colab المجاني بيفصل بعد فترة خمول أو طول جلسة. الأدابتور بيضيع لو الفصل حصل قبل خلية الرفع (8) — عشان كده أول تشغيلة صغيرة (3000 مثال). للتشغيلات الأكبر: قسّم الداتا على كذا جلسة، أو فعّل حفظ checkpoint كل فترة وارفع كل مرحلة. |
| **`HF_TOKEN` not found / 401 / 403** | اتأكد إن التوكن من نوع **Write**، وإنه موجود في Colab Secrets باسم `HF_TOKEN` بالظبط، وإن "Notebook access" مفعّلة للسر ده. |
| **Repository not found عند الرفع** | ده طبيعي لو الريبو مش موجود — `push_to_hub` بينشئه تلقائياً طالما التوكن Write واسم المستخدم صح. |
| **النموذج بيرد رد ضعيف بعد التدريب** | متوقّع جزئياً مع 3000 مثال و epoch واحدة — دي تشغيلة إثبات فكرة. زوّد `SAMPLE_SIZE` و `NUM_EPOCHS` لـ 2 في التشغيلات الجاية وهتلاحظ الفرق. |

## Next steps — اللي بعده
1. جرّب الأدابتور محلياً بـ `inference.py` اللي في الريبو.
2. زوّد حجم الداتا تدريجياً (10K ثم أكتر) في جلسات منفصلة.
3. ادمج الأدابتور في النموذج (merge) أو صدّره GGUF عشان يشتغل محلياً بأدوات زي Ollama / llama.cpp — شوف قسم Roadmap في README.